<a href="https://colab.research.google.com/github/kligeys/AI-Projects/blob/main/Rag_company_benefits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install faiss-gpu

In [10]:
import numpy as np
import faiss
from google.colab import userdata
from openai import OpenAI

In [11]:
# 1. Retrieve the API key safely from Colab Secrets
api_key = userdata.get('OPENAI_API_KEY')

In [12]:
# 2. Initialize the OpenAI client
client = OpenAI(api_key=api_key)

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
drive_path = '/content/drive/MyDrive/Colab Notebooks/Benefit.txt'

with open(drive_path, 'r', encoding='utf-8') as file:
    content = file.read()
    #print(content)

In [17]:
# 2. Split text into simple chunks (by sentence or paragraphs)
chunks = [chunk.strip() for chunk in content.split("\n") if chunk.strip()]


In [18]:
# 3. Generate embeddings for each chunk using OpenAI
def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(input=[text], model=model)
    return response.data[0].embedding

In [20]:
chunk_embeddings = [get_embedding(chunk) for chunk in chunks]
embeddings_array = np.array(chunk_embeddings).astype("float32")

In [21]:
# 4. Store embeddings in a FAISS vector index
dimension = embeddings_array.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_array)

In [23]:
# 5. Define retrieval and generation function
def ask_rag(query, k=2):
    # Embed the user query
    query_embedding = np.array([get_embedding(query)]).astype("float32")

    # Search top-k similar chunks
    distances, indices = index.search(query_embedding, k)
    retrieved_chunks = [chunks[i] for i in indices[0]]

    # Construct context-aware prompt for ChatGPT
    context = "\n".join(retrieved_chunks)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer based only on the context:"

    # Call ChatGPT (using gpt-4o-mini or gpt-4o)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer questions strictly using the provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content, retrieved_chunks

In [29]:
# 6. Test the RAG system
query = "How much is the 401k match amount in dollars"
answer, sources = ask_rag(query)

In [30]:
print("--- Answer ---")
print(answer)
print("\n--- Retrieved Sources ---")
for s in sources:
    print(f"- {s}")

--- Answer ---
The 401(k) match amount is up to $12,250.

--- Retrieved Sources ---
- * 401(k) Match: A 50% employer match on employee contributions up to half of the IRS annual limit (e.g., matching up to $12,250).
- * Charitable Matching: 100% match on employee charitable donations up to $10,000 per year.
